In [1]:
import emoji
import nltk
import numpy as np
import pandas as pd
import re

from nltk.tokenize import TweetTokenizer
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer

Change the path for the csv ***HERE*** accordingly to load/save

In [2]:
IN_CSV = "../../data/twitter_training.csv"
OUT_CSV = '../../data/output_file.csv'

TITLES = ["id", "entity", "sentiment", "text"]

In [3]:
df = pd.read_csv(IN_CSV, names=TITLES).astype(str)

In [4]:
df.head()

,id,entity,sentiment,text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...


In [5]:
tokenizer = TweetTokenizer()

Bingxi: Note for myself that twitter usernames are not case sensitive.

In [19]:
def preprocess(row):
    entity = row["entity"]
    text = row["text"]

    FLAG = re.I

    emoji_pattern = r"(\ud83c[d000-dfff]|\ud83d[d000-dfff]|\ud83e[d000-dfff])"

    url_pattern = r'(https?:\/\/)?(([\w\-]{1,63}\.)+[\w\-]{1,63})(\/[\w.\-]+)*([?=+%\w~\-.&]*)'
    entity_pattern = rf"@?{entity}"
    user_pattern = r"@\w+"
    repeated_pattern = r"(.)\1\1\1+"
    repeated_replace_pattern = r"\1\1\1"

    # Replaces emoji
    # emojis = re.findall(emoji_pattern, text)
    
    # if len(emojis) != 0:
    #     print(f"{row.name}: {emojis}")
    #     for e in emojis:
    #         if not emoji.is_emoji(e):
    #             continue
    #         e_meaning = emoji.demojize(e)[1:-1]
    #         e_replacement = f"<EMOJI_{e_meaning.upper()}>"
    #         print(f"{e} -> {e_replacement}")
    #         text = re.sub(rf"{e}", e_replacement, text)
    #     print("\n")


    # Concate if potential URL/Username
    text = re.sub(r"\s+\/(\s+)?", "/", text)
    text = re.sub(r"\@\s+", "@", text)

    # Replaces URL
    text = re.sub(url_pattern, '<URL>', text)

    # Replaces entity and other_user
    # text = re.sub(r"@ *")
    text = re.sub(entity_pattern, "<ENTITY>", text, flags=FLAG)
    text = re.sub(user_pattern, "<OTHER_USER>", text)

    # Raplaces repeated patterns
    text = re.sub(repeated_pattern, repeated_replace_pattern, text)
    
    tokens = tokenizer.tokenize(text)

    for token in tokens:
        if not emoji.is_emoji(token):
            continue

        print(f"{row.name}: {token}")
        e_meaning = emoji.demojize(token)[1:-1]
        e_replacement = f"<EMOJI_{e_meaning.upper()}>"
        print(f"{token} -> {e_replacement}")
        token = e_replacement
    
    return tokens

In [ ]:
df["cleaned_text"] = df.apply(preprocess, axis=1)

78: ‼
‼ -> <EMOJI_DOUBLE_EXCLAMATION_MARK>
81: ‼
‼ -> <EMOJI_DOUBLE_EXCLAMATION_MARK>
222: 1⃣
1⃣ -> <EMOJI_KEYCAP_1>
225: 1⃣
1⃣ -> <EMOJI_KEYCAP_1>
576: 🤔
🤔 -> <EMOJI_THINKING_FACE>
579: 🤔
🤔 -> <EMOJI_THINKING_FACE>
648: 🤔
🤔 -> <EMOJI_THINKING_FACE>
651: 🤔
🤔 -> <EMOJI_THINKING_FACE>
652: 🤔
🤔 -> <EMOJI_THINKING_FACE>
672: 🥰
🥰 -> <EMOJI_SMILING_FACE_WITH_HEARTS>
672: 🥰
🥰 -> <EMOJI_SMILING_FACE_WITH_HEARTS>
672: 🥰
🥰 -> <EMOJI_SMILING_FACE_WITH_HEARTS>
708: 🤞
🤞 -> <EMOJI_CROSSED_FINGERS>
711: 🤞
🤞 -> <EMOJI_CROSSED_FINGERS>
894: 🧨
🧨 -> <EMOJI_FIRECRACKER>
894: 🧨
🧨 -> <EMOJI_FIRECRACKER>
897: 🧨
🧨 -> <EMOJI_FIRECRACKER>
897: 🧨
🧨 -> <EMOJI_FIRECRACKER>
1074: 🧘
🧘 -> <EMOJI_PERSON_IN_LOTUS_POSITION>
1074: 🧘
🧘 -> <EMOJI_PERSON_IN_LOTUS_POSITION>
1077: 🧘
🧘 -> <EMOJI_PERSON_IN_LOTUS_POSITION>
1077: 🧘
🧘 -> <EMOJI_PERSON_IN_LOTUS_POSITION>
1078: 🧘
🧘 -> <EMOJI_PERSON_IN_LOTUS_POSITION>
1078: 🧘
🧘 -> <EMOJI_PERSON_IN_LOTUS_POSITION>
1128: 🤔
🤔 -> <EMOJI_THINKING_FACE>
1131: 🤔
🤔 -> <EMOJI_THINKING_FACE>
1

Bingxi: 
    This is for me to test whether my code for replacing emoji is working, 
    new implementation using "emoji" package is testing

In [13]:
df.iloc[576]

id                                                           2500
entity                                                Borderlands
sentiment                                              Irrelevant
text            Top 4 favourite games you say? 🤔. . Sea of Thi...
cleaned_text    [Top, 4, favourite, games, you, say, ?, 🤔, . ....
Name: 576, dtype: object

In [14]:
df.head()

,id,entity,sentiment,text,cleaned_text
0,2401,Borderlands,Positive,im getting on borderlands and i will murder yo...,"[im, getting, on, <ENTITY>, and, i, will, murd..."
1,2401,Borderlands,Positive,I am coming to the borders and I will kill you...,"[I, am, coming, to, the, borders, and, I, will..."
2,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...,"[im, getting, on, <ENTITY>, and, i, will, kill..."
3,2401,Borderlands,Positive,im coming on borderlands and i will murder you...,"[im, coming, on, <ENTITY>, and, i, will, murde..."
4,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...,"[im, getting, on, <ENTITY>, 2, and, i, will, m..."


In [15]:
df.to_csv(OUT_CSV)